# 03 — Exploratory Data Analysis

The tables built in notebook 02 are clean; here the work shifts from *shaping* the data to
*understanding* it. Several passes over the hourly electricity data, plus a first look at
the annual series — each surfacing a hook for a later research question.

| Pass | What | Feeds |
|---|---|---|
| Completeness | gaps & nulls, proven with a window-function scan | trust in every RQ |
| Distributions | robust summary stats first; plot only what a number can't show | RQ4 (price regime) |
| Seasonal & diurnal | demand by hour/weekday/month, solar summer-vs-winter, temp arc | RQ2, RQ3 |
| Annual first-look | GHG trajectory + sector decomposition | RQ6 |

Two principles carried throughout:

- **Numbers before pictures.** A summary table that says everything makes a plot redundant;
  a figure earns its place only when it shows something a statistic can't (here, the 2022
  price regime shift).
- **Local time only at the edge.** Every clock-dependent aggregation converts
  `ts_utc → Europe/Vienna` at query time — a diurnal profile on raw UTC is shifted 1–2 h and
  smears the DST fall-back hour.

Uses the shared plotting helpers in `src/viz.py` (`set_house_style()`, `PALETTE`,
`line_profile()`). Reads `data/processed/austria_energy.duckdb` **read-only** — the database
is built by notebook 02.

## Setup

In [ ]:
# imports, paths, house style, read-only DuckDB connection
import sys
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.viz import set_house_style, PALETTE, line_profile

set_house_style()

DB_PATH = PROJECT_ROOT / "data" / "processed" / "austria_energy.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)


## 1 · Missingness & continuity

In [ ]:
# In a time series, a missing hour is an ABSENT ROW, not a null.
# expected_hours = span in hours + 1 (inclusive). If actual == expected and
# there are no gaps, the index is continuous.
# NOTE: f-string SQL here is fine because table/column names are hardcoded
# constants — never do this with user-supplied input (SQL injection).
def continuity_check(con, table: str, value_col: str) -> pd.DataFrame:
    return con.sql(f"""
        SELECT
            '{table}'                                          AS tbl,
            COUNT(*)                                           AS rows,
            date_diff('hour', MIN(ts_utc), MAX(ts_utc)) + 1    AS expected_hours,
            date_diff('hour', MIN(ts_utc), MAX(ts_utc)) + 1
                - COUNT(*)                                     AS missing_hours,
            COUNT(*) FILTER (WHERE {value_col} IS NULL)        AS null_values
        FROM {table}
    """).df()

checks = [("demand", "demand_mw"), ("prices", "price_eur_mwh")]
display(pd.concat([continuity_check(con, t, c) for t, c in checks],
                  ignore_index=True))

# Weather has 5 value columns — scan each
display(con.sql("""
SELECT
    COUNT(*)                                         AS rows,
    date_diff('hour', MIN(ts_utc), MAX(ts_utc)) + 1  AS expected_hours,
    COUNT(*) FILTER (WHERE temperature_c  IS NULL)   AS null_temp,
    COUNT(*) FILTER (WHERE solar_wm2      IS NULL)   AS null_solar,
    COUNT(*) FILTER (WHERE wind_kmh       IS NULL)   AS null_wind,
    COUNT(*) FILTER (WHERE precip_mm      IS NULL)   AS null_precip,
    COUNT(*) FILTER (WHERE cloudcover_pct IS NULL)   AS null_cloud
FROM weather
""").df())

In [ ]:
# explicitly locate any gap using a window function.
# LAG pulls the previous row's timestamp; any step ≠ 1 hour is a gap.
# (This is the first window function — the OVER(ORDER BY) pattern is the
#  the later duck-curve and merit-order analyses.)
con.sql("""
WITH stepped AS (
    SELECT
        ts_utc,
        ts_utc - LAG(ts_utc) OVER (ORDER BY ts_utc) AS step
    FROM demand
)
SELECT ts_utc AS gap_after, step
FROM stepped
WHERE step <> INTERVAL 1 HOUR     -- first row's step is NULL, auto-excluded
ORDER BY ts_utc
""").df()   # expect 0 rows

Continuity verified in UTC, no imputation needed

## 2 · Summary statistics

In [ ]:
# median + p01/p99 are robust to the 2022 price spike in a way mean/std aren't.
# pct_zero is the zero-inflation flag (watch solar & precip light up).
con.sql("""
WITH series AS (
    SELECT 'demand_mw'  AS metric, demand_mw     AS v FROM demand
    UNION ALL SELECT 'price_eur_mwh', price_eur_mwh FROM prices
    UNION ALL SELECT 'temp_c',        temperature_c  FROM weather
    UNION ALL SELECT 'solar_wm2',     solar_wm2      FROM weather
    UNION ALL SELECT 'wind_kmh',      wind_kmh       FROM weather
    UNION ALL SELECT 'precip_mm',     precip_mm      FROM weather
    UNION ALL SELECT 'cloud_pct',     cloudcover_pct FROM weather
)
SELECT
    metric,
    ROUND(AVG(v), 1)                       AS mean,
    ROUND(median(v), 1)                    AS p50,
    ROUND(stddev(v), 1)                    AS std,
    ROUND(MIN(v), 1)                       AS min,
    ROUND(quantile_cont(v, 0.99), 1)       AS p99,
    ROUND(MAX(v), 1)                       AS max,
    ROUND(skewness(v), 2)                  AS skew,
    ROUND(100.0 * COUNT(*) FILTER (WHERE v = 0) / COUNT(*), 1) AS pct_zero
FROM series
GROUP BY metric
ORDER BY metric
""").df()

In [ ]:
# generation per fuel — the 8 REAL fuels only.
# Excludes the 5 nominal/placeholder fuels (constant or zero output).
# peak_to_avg surfaces solar's ~19× ratio → this is the RQ3 duck-curve hook.
con.sql("""
SELECT
    fuel_type,
    ROUND(AVG(mw))                              AS mean_mw,
    ROUND(median(mw))                           AS p50_mw,
    ROUND(MIN(mw))                              AS min_mw,
    ROUND(MAX(mw))                              AS max_mw,
    ROUND(100.0 * COUNT(*) FILTER (WHERE mw = 0) / COUNT(*), 1) AS pct_zero,
    ROUND(MAX(mw) / NULLIF(AVG(mw), 0), 1)      AS peak_to_avg
FROM generation
WHERE flow = 'generation'
  AND fuel_type NOT IN ('Waste','Other','Geothermal','Fossil Oil','Other renewable')
GROUP BY fuel_type
ORDER BY mean_mw DESC
""").df()

## 3 · Price floor & negative prices

In [ ]:
# Floor integrity check. A hard floor means: min == floor, a pile-up AT the
# floor, and ZERO observations below it.
# Note: comparing a DOUBLE with a tolerance (<= -499.99) instead of `= -500.0`
# is the safe habit — float equality is fragile in general, even though -500
# happens to be exactly representable here.
con.sql("""
SELECT
    MIN(price_eur_mwh)                                AS min_price,
    COUNT(*) FILTER (WHERE price_eur_mwh <= -499.99)  AS at_floor,
    COUNT(*) FILTER (WHERE price_eur_mwh <  -500.01)  AS below_floor,   -- expect 0
    COUNT(*) FILTER (WHERE price_eur_mwh <  0)        AS negative_hours
FROM prices
""").df()

In [ ]:
# Floor hours in Vienna local time — they should cluster in sunny/windy,
# low-demand hours (spring/summer weekend middays). That's the merit-order
# floor: renewable oversupply with nowhere to go.
con.sql("""
SELECT
    ts_utc AT TIME ZONE 'Europe/Vienna'  AS ts_local,
    price_eur_mwh
FROM prices
WHERE price_eur_mwh <= -499.99
ORDER BY ts_local
""").df()

In [ ]:
# When do negative prices happen? If it's the merit-order effect, they cluster
# at midday (solar oversupply). Hour-of-day computed in Vienna LOCAL time —
# this is the seasonal-aggregation pattern we'll lean on throughout the later analyses.
con.sql("""
SELECT
    EXTRACT(hour FROM ts_utc AT TIME ZONE 'Europe/Vienna')  AS hour_local,
    COUNT(*) FILTER (WHERE price_eur_mwh < 0)               AS negative_hours,
    ROUND(AVG(price_eur_mwh), 1)                            AS avg_price
FROM prices
GROUP BY hour_local
ORDER BY hour_local
""").df()

## 4 · Price distribution — the 2022 regime shift

In [ ]:
# the ONE distribution worth plotting
# Of 13 metrics, only prices show a regime shift a summary number can't convey.

# Per-year price structure — robust quantiles computed in SQL.
# Year extracted in Vienna local time; the offset is immaterial at annual grain.
yr = con.sql("""
SELECT
    EXTRACT(year FROM ts_utc AT TIME ZONE 'Europe/Vienna')  AS yr,
    quantile_cont(price_eur_mwh, 0.25)                 AS p25,
    median(price_eur_mwh)                              AS p50,
    quantile_cont(price_eur_mwh, 0.75)                 AS p75,
    quantile_cont(price_eur_mwh, 0.99)                 AS p99,
    MAX(price_eur_mwh)                                 AS max,
    100.0 * COUNT(*) FILTER (WHERE price_eur_mwh < 0)
          / COUNT(*)                                   AS pct_neg
FROM prices
WHERE EXTRACT(year FROM ts_utc AT TIME ZONE 'Europe/Vienna') <= 2024
GROUP BY yr
ORDER BY yr
""").df()

fig, ax = plt.subplots()
x = yr["yr"]

ax.vlines(x, yr["p25"], yr["p75"], color=PALETTE["price"], lw=9, alpha=0.85,
          label="IQR (p25–p75)")
ax.vlines(x, yr["p75"], yr["p99"], color=PALETTE["price"], lw=2, alpha=0.45,
          label="p75 → p99")
ax.scatter(x, yr["p50"], facecolor="white", edgecolor=PALETTE["price"],
           s=45, zorder=5, label="median")
ax.scatter(x, yr["max"], marker="x", color=PALETTE["accent"], s=55, zorder=5,
           label="max (worst single hour)")

ax.axhline(0, color=PALETTE["muted"], lw=0.8)   # negative prices live below this
ax.set_ylabel("day-ahead price (€/MWh)")
ax.set_title("Austria day-ahead prices by year — the 2022 regime shift")
ax.legend(loc="upper left", ncol=2)
plt.show()

display(yr.round(1))

In [ ]:
con.sql("""
SELECT 'prices'  AS tbl, MIN(ts_utc) AS lo, MAX(ts_utc) AS hi, COUNT(*) AS n FROM prices
UNION ALL
SELECT 'demand',         MIN(ts_utc),       MAX(ts_utc),       COUNT(*)       FROM demand
UNION ALL
SELECT 'weather',        MIN(ts_utc),       MAX(ts_utc),       COUNT(*)       FROM weather
ORDER BY tbl
""").df()

## 5 · Demand seasonality

In [ ]:
# At three grains
# Same recipe each time: convert to Vienna local → EXTRACT a time part → average.
# Local time is non-negotiable here: a diurnal profile on raw UTC is shifted 1–2h.

def demand_profile(part: str):
    # `part` is a trusted constant (hour/isodow/month) → f-string is safe.
    return con.sql(f"""
        SELECT EXTRACT({part} FROM ts_utc AT TIME ZONE 'Europe/Vienna') AS x,
               AVG(demand_mw)   AS avg_demand
        FROM demand
        GROUP BY x ORDER BY x
    """).df()

hour  = demand_profile("hour")
week  = demand_profile("isodow")    # 1 = Mon … 7 = Sun
month = demand_profile("month")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].plot(hour["x"], hour["avg_demand"], color=PALETTE["demand"], marker="o", ms=3)
ax[0].set(title="By hour of day (local)", xlabel="hour", ylabel="avg demand (MW)")

ax[1].plot(week["x"], week["avg_demand"], color=PALETTE["demand"], marker="o", ms=4)
ax[1].set(title="By day of week")
ax[1].set_xticks(range(1, 8)); ax[1].set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])

ax[2].plot(month["x"], month["avg_demand"], color=PALETTE["demand"], marker="o", ms=4)
ax[2].set(title="By month")
ax[2].set_xticks(range(1, 13)); ax[2].set_xticklabels(list("JFMAMJJASOND"))

fig.suptitle("Austrian electricity demand — seasonal profiles", y=1.02)
plt.tight_layout()
plt.show()

## 6 · Solar diurnal shape

In [ ]:
# Summer vs winter
# The RQ3 duck-curve driver. FILTER (familiar) splits one query into two seasonal
# columns; the CTE converts to local time once.
solar = con.sql("""
WITH solar_local AS (
    SELECT
        ts_utc AT TIME ZONE 'Europe/Vienna' AS loc,   -- convert once, reuse below
        mw
    FROM generation
    WHERE flow = 'generation' AND fuel_type = 'Solar'
)
SELECT
    EXTRACT(hour FROM loc)                                        AS hour_local,
    AVG(mw) FILTER (WHERE EXTRACT(month FROM loc) IN (6, 7, 8))   AS summer_mw,
    AVG(mw) FILTER (WHERE EXTRACT(month FROM loc) IN (12, 1, 2))  AS winter_mw
FROM solar_local
GROUP BY hour_local
ORDER BY hour_local
""").df()

fig, ax = plt.subplots()
line_profile(ax, solar["hour_local"], solar["summer_mw"],
             color=PALETTE["solar"], label="summer (Jun–Aug)")
line_profile(ax, solar["hour_local"], solar["winter_mw"],
             color=PALETTE["muted"], label="winter (Dec–Feb)")
ax.set(title="Solar generation by hour — summer vs winter",
       xlabel="hour (local)", ylabel="avg solar (MW)")
ax.legend()
plt.show()

display(solar.round(0))

## 7 · Temperature by month

In [ ]:
# The RQ2 hook
temp = con.sql("""
SELECT
    EXTRACT(month FROM ts_utc AT TIME ZONE 'Europe/Vienna') AS month,
    AVG(temperature_c) AS avg_temp
FROM weather
GROUP BY month
ORDER BY month
""").df()

fig, ax = plt.subplots()
line_profile(ax, temp["month"], temp["avg_temp"], color=PALETTE["temp"])
ax.set(title="Average temperature by month", xlabel="", ylabel="avg temperature (°C)")
ax.set_xticks(range(1, 13)); ax.set_xticklabels(list("JFMAMJJASOND"))
plt.show()

display(temp.round(1))

## 8 · GHG emissions trajectory

In [ ]:
# Trajectory + sector mix
# The stacked emitting sectors sum to the total (excl. LULUCF); the black line overlays
# that total as a sanity check and traces the trajectory.
# NOTE: LULUCF (CRF4) is a *sink* (negative) — excluded from the emissions stack.

# 5 emitting top-level sectors (exclude LULUCF + the totals)
ghg = con.sql("""
SELECT year, sector, mt_co2e
FROM ghg_emissions
WHERE sector_code IN ('CRF1','CRF2','CRF3','CRF5','CRF6')
ORDER BY year
""").df()

# headline total, to overlay
total = con.sql("""
SELECT year, mt_co2e
FROM ghg_emissions
WHERE sector_code = 'TOTX4_MEMO'
ORDER BY year
""").df()

# long → wide for stackplot, ordered largest → smallest for readability
wide  = ghg.pivot(index="year", columns="sector", values="mt_co2e")
order = wide.mean().sort_values(ascending=False).index.tolist()
wide  = wide[order]

sector_colors = {
    "Energy":               PALETTE["accent"],    # coral  — dominant sector
    "Industrial processes": PALETTE["price"],     # purple
    "Agriculture":          PALETTE["wind"],      # teal
    "Waste":                PALETTE["demand"],    # grey
    "Other":                PALETTE["muted"],
}

fig, ax = plt.subplots(figsize=(12, 5))
ax.stackplot(wide.index, [wide[s] for s in order], labels=order,
             colors=[sector_colors[s] for s in order], alpha=0.9)
ax.plot(total["year"], total["mt_co2e"], color="black", lw=1.2,
        label="Total (excl. LULUCF)")
ax.axvline(2005, color=PALETTE["muted"], ls="--", lw=1)   # the target baseline year
ax.set_title("Austria GHG emissions by sector (excl. LULUCF), 1990–2024")
ax.set_ylabel("Mt CO₂-eq")
ax.set_xlim(wide.index.min(), wide.index.max())
ax.legend(loc="lower left", ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

# quick numeric read: total vs the 2005 baseline
base2005   = total.set_index("year").loc[2005, "mt_co2e"]
total_pct  = total.assign(pct_vs_2005=((total["mt_co2e"] / base2005 - 1) * 100).round(1))
display(total_pct.tail(8))

## 9 · ESR non-ETS structure

In [ ]:
# structural EDA of the ESR series
# (i) span + where the accounting regime switches (ESD/AR4 → ESR/AR5)
con.sql("""
SELECT regime,
       MIN(year) AS first_year,
       MAX(year) AS last_year,
       COUNT(*)  AS n_years
FROM esr_emissions
GROUP BY regime
ORDER BY first_year
""").df()

In [ ]:
# baseline, change vs 2005, YoY delta, and
# the ESR slice as a share of the national total (TOTX4_MEMO, excl. LULUCF).
con.sql("""
WITH base AS (                       -- 2005 reference (ESD-basis in this file)
    SELECT mt_co2e AS base_2005
    FROM esr_emissions
    WHERE year = 2005
),
total AS (                           -- national total excl. LULUCF
    SELECT year, mt_co2e AS total_mt
    FROM ghg_emissions
    WHERE sector_code = 'TOTX4_MEMO'
)
SELECT
    e.year,
    e.regime,
    ROUND(e.mt_co2e, 2)                                        AS esr_mt,
    ROUND(100.0 * (e.mt_co2e / b.base_2005 - 1), 1)            AS pct_vs_2005,
    ROUND(e.mt_co2e - LAG(e.mt_co2e) OVER (ORDER BY e.year), 2) AS yoy_delta,
    ROUND(100.0 * e.mt_co2e / t.total_mt, 1)                   AS share_of_total_pct
FROM esr_emissions e
CROSS JOIN base b
LEFT JOIN total t USING (year)
ORDER BY e.year
""").df()

In [ ]:
# Close the connection
con.close()

## Summary

EDA covered completeness, distributions, and seasonal structure across the hourly data
(demand, prices, weather, generation), plus a first look at the annual GHG inventory. Every
finding is a hook for its research question.

- **Completeness.** Every hourly bucket present, zero nulls, zero gaps (verified in UTC via
  row-count-vs-expected and a `LAG` gap scan). No imputation needed.
- **Prices → RQ4.** A 2022 regime shift (median ~€224 vs ~€35 in 2019–20), heavily
  right-skewed. Negative prices ~1.2 % of hours (rising to 3.4 % in 2024), clustered at midday
  (solar oversupply) with a smaller overnight mode. The −500 €/MWh floor binds once.
- **Demand → RQ2.** Double-humped day (morning + evening peaks), weekday > weekend,
  winter-peak / summer-trough. Heating-driven — no summer cooling spike (little AC in AT).
- **Temperature → RQ2.** Warm-summer / cold-winter arc (~2 °C Jan, ~22 °C Jul) — a near
  inverse mirror of demand-by-month.
- **Solar → RQ3.** Summer midday avg ~1025 MW vs winter ~320 MW (~3×), the summer bump wider.
  The duck curve is effectively a summer phenomenon — winter solar barely dents net load.
- **Generation mix → RQ1.** Hydro run-of-river is the baseload backbone; gas is the flexible
  price-setter; coal ~phased out (2020); biomass near-constant (must-run). The 5 nominal/zero
  fuels are excluded from variability work.
- **GHG → RQ6.** Total (excl. LULUCF) peaked ~93 Mt in 2005, fell to 66.6 Mt in 2024
  (≈ −28 % vs 2005), most of the drop after 2019. Energy (CRF1) dominates and declines most.

**Next:** the research-question notebooks (`04_rq1_…` through `09_rq6_…`), each mixing SQL,
`statsmodels`, and `matplotlib` and ending with a headline visual and a two-sentence finding.